# iTransformer + Composite Loss + Deep Ensemble + Adaptive Conformal

Эксперимент по предложенному SOTA-стеку для краткосрочного прогноза направления цены на 5-мин барах MOEX. Все 4 компоненты собраны вместе:

1. **iTransformer** (Liu et al., ICLR 2024, [arXiv:2310.06625](https://arxiv.org/abs/2310.06625)) — inverted attention поверх variate-токенов. Решает межканальную зависимость, которой не хватало TimeXer'у.
2. **CompositeQuantLoss** = α·BCE + β·RankIC + γ·Sharpe + δ·Monotone. Прямая оптимизация целевых метрик стратегии (Microsoft Qlib + Lim-Zohren-Roberts 2019).
3. **Deep Ensemble (M=5)** (Lakshminarayanan et al., NeurIPS 2017, [arXiv:1612.01474](https://arxiv.org/abs/1612.01474)) — заменяет MC Dropout как источник эпистемической неопределённости.
4. **Adaptive Conformal Inference** (Gibbs & Candès, 2021, [arXiv:2106.00170](https://arxiv.org/abs/2106.00170)) — формальная гарантия покрытия при distribution shift, адаптируется онлайн.

Базируется на тех же ALGOPACK-данных (TradeStats / OrderStats / OBStats), что и `algopack_baseline.ipynb`.

## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH],
                       check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard',
                        f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
             f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
            check=True, env=env,
        )
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}, IN_COLAB: {IN_COLAB}')

In [ ]:
if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '--quiet',
         'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
         'scikit-learn>=1.4', 'requests>=2.31', 'tqdm>=4.66', 'ipywidgets>=8.0'],
        check=True,
    )
    print('Готово.')

In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if not (SRC / 'graduate_work' / '__init__.py').exists():
    raise FileNotFoundError(f'graduate_work не найден в {SRC}')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from graduate_work.config import default_config, ModelConfig, TradingConfig

cfg = default_config()
# Переопределяем архитектуру и loss-objective под этот эксперимент.
cfg = dataclasses.replace(
    cfg,
    model=dataclasses.replace(
        cfg.model,
        architecture='itransformer',
        itransformer_seq_len=cfg.data.window_size,
    ),
    trading=dataclasses.replace(cfg.trading, loss_objective='composite'),
)
print('architecture =', cfg.model.architecture)
print('loss_objective =', cfg.trading.loss_objective)
print('Тикеры:', cfg.data.tickers)
print('Горизонты:', cfg.data.horizons)
print('Окно:', cfg.data.window_size)

## 1. Drive: данные ALGOPACK

Ожидается такая же структура, что и у `algopack_baseline.ipynb`: SuperCandles + календари + дивиденды в `MyDrive/lstm_exchange/data/raw/Algopack/`.

In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Подтянули данные из {src_dir}')
    else:
        raise FileNotFoundError(f'Не найдено {src_dir}.')

for sub in ['algopack/tradestats', 'algopack/orderstats', 'algopack/obstats',
            'calendars', 'dividends']:
    p = cfg.paths.data_raw / sub
    if p.exists():
        n = sum(1 for _ in p.glob('*.csv'))
        print(f'  {sub}: {n} файлов')
    else:
        print(f'  {sub}: ОТСУТСТВУЕТ')

## 2. Загрузка SuperCandles

In [ ]:
def _load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def _load_supercandle(product: str, ticker: str) -> pd.DataFrame:
    p = cfg.paths.data_raw / 'algopack' / product / f'{ticker}.csv'
    df = _load_csv(p)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(
            df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
            utc=True, errors='coerce',
        )
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts
    df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


ts_data, os_data, ob_data = {}, {}, {}
div_data = {}
for ticker in cfg.data.tickers:
    ts_data[ticker] = _load_supercandle('tradestats', ticker)
    os_data[ticker] = _load_supercandle('orderstats', ticker)
    ob_data[ticker] = _load_supercandle('obstats', ticker)
    p = cfg.paths.data_raw / 'dividends' / f'{ticker}.csv'
    if p.exists():
        df = pd.read_csv(p)
        if 'registryclosedate' in df.columns:
            df['registryclosedate'] = pd.to_datetime(
                df['registryclosedate'], utc=True, errors='coerce',
            )
        div_data[ticker] = df

cal_dir = cfg.paths.data_raw / 'calendars'
calendars = {p.stem: pd.read_csv(p) for p in cal_dir.glob('*.csv')} if cal_dir.exists() else {}

for t, df in ts_data.items():
    print(f'  {t}: {len(df)} 5-мин баров, range {df.index.min()} .. {df.index.max() if not df.empty else "-"}')

## 3. Build feature frame (TradeStats backbone + микроструктура + календари + дивиденды)

In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features, order_to_trade_ratio,
)
from graduate_work.features.calendar_features import (
    dividend_features, trading_day_features, expirations_features,
)
from graduate_work.features.targets import cost_aware_classification_labels, lr_columns


def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    if ts.empty:
        return pd.DataFrame()
    out = pd.DataFrame(index=ts.index)
    out['open'] = ts['pr_open'].astype(float)
    out['high'] = ts['pr_high'].astype(float)
    out['low']  = ts['pr_low'].astype(float)
    out['close']= ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


def build_ticker_frame(ticker: str) -> pd.DataFrame:
    ts_df = ts_data[ticker]
    if ts_df.empty:
        return pd.DataFrame()
    feat = _ts_to_ohlcv(ts_df, ticker)
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    feat = feat.join(tradestats_features(ts_df), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty and not ts_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    trading_days = calendars.get('trading_days_stock', pd.DataFrame())
    feat = feat.join(trading_day_features(trading_days, feat.index), how='left')
    expir = calendars.get('futures_expirations', pd.DataFrame())
    feat = feat.join(expirations_features(expir, feat.index), how='left')
    if ticker in div_data and not div_data[ticker].empty:
        feat = feat.join(
            dividend_features(div_data[ticker], feat.index, last_close_price=None),
            how='left',
        )
    distance_cols = [c for c in feat.columns if c.startswith(('cal_days_', 'div_days_'))]
    if distance_cols:
        feat[distance_cols] = feat[distance_cols].ffill()
    return feat.fillna(0.0)


frames = []
for ticker in cfg.data.tickers:
    f = build_ticker_frame(ticker)
    if not f.empty:
        frames.append(f)
        print(f'  {ticker}: {f.shape}')

full = pd.concat(frames).sort_index()
print(f'\nfull frame: {full.shape}')

## 4. Targets (cost-aware classification) + lr-arrays для composite loss

In [ ]:
def _attach_targets_and_lr(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    out_parts = []
    for ticker, sub in frame.groupby('ticker'):
        labels = cost_aware_classification_labels(
            open_price=sub['open'], close_price=sub['close'],
            horizons=cfg.data.horizons,
            entry_cost=cfg.trading.commission_rate + cfg.trading.slippage_rate,
            exit_cost=cfg.trading.commission_rate + cfg.trading.slippage_rate,
            label_smoothing=cfg.data.label_smoothing,
            direction='long',
        )
        out_parts.append(sub.join(labels, how='left'))
    out = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')
    target_cols = [f'target_h{h}' for h in cfg.data.horizons]
    lr_cols = lr_columns(cfg.data.horizons)
    return out, target_cols, lr_cols


full_with_targets, target_cols, lr_cols = _attach_targets_and_lr(full)
print(f'frame: {full_with_targets.shape}')
print(f'target_cols: {target_cols}')
print(f'lr_cols: {lr_cols}')
for c in target_cols:
    p_up = float((full_with_targets[c].dropna() >= 0.5).mean())
    print(f'  {c}: P(UP)={p_up:.3f}')

feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and c not in lr_cols]
print(f'\nfeature_cols: {len(feature_cols)} (без targets и lr_h*)')

In [ ]:
from graduate_work.features.pipeline import chronological_split, _build_arrays
from graduate_work.features.scaler import StandardScaler
from graduate_work.strategy import extract_lr_array

train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio, cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)
test_prices_raw = test_df[['open', 'close', 'ticker']].copy()

scaler = StandardScaler()
scaler.fit(train_df, feature_cols)
train_df_s = scaler.transform(train_df)
val_df_s   = scaler.transform(val_df)
test_df_s  = scaler.transform(test_df)

train = _build_arrays(train_df_s, feature_cols, target_cols, cfg.data.window_size, desc='train')
val   = _build_arrays(val_df_s,   feature_cols, target_cols, cfg.data.window_size, desc='val')
test  = _build_arrays(test_df_s,  feature_cols, target_cols, cfg.data.window_size, desc='test')

# lr-arrays — сырые лог-доходности per-window под composite loss.
# Берём из НЕскейленного train_df/val_df (lr_h* — это сырые числа,
# их нельзя стандартизировать).
train_lr = extract_lr_array(train_df, train, cfg.data.horizons)
val_lr   = extract_lr_array(val_df,   val,   cfg.data.horizons)
print(f'train: x={train["x"].shape}, y={train["y"].shape}, lr={train_lr.shape}')
print(f'val:   x={val["x"].shape},   y={val["y"].shape},   lr={val_lr.shape}')
print(f'test:  x={test["x"].shape},  y={test["y"].shape}')

## 5. Deep Ensemble обучения iTransformer + composite loss

M=5 моделей с разными сидами. Каждая — свой локальный минимум, разница между ними даёт эпистемическую неопределённость без MC Dropout.

In [ ]:
from graduate_work.model import build_model
from graduate_work.training import DeepEnsembleTrainer, ensemble_predict

ENSEMBLE_SIZE = 5
INPUT_DIM = train['x'].shape[-1]
NUM_HORIZONS = len(cfg.data.horizons)

def model_factory(seed: int):
    return build_model(input_dim=INPUT_DIM, num_horizons=NUM_HORIZONS, cfg=cfg.model)

ckpt_dir = cfg.paths.checkpoints / 'itransformer_ensemble'
ensemble = DeepEnsembleTrainer(
    model_factory,
    cfg.training,
    ensemble_size=ENSEMBLE_SIZE,
    data_cfg=cfg.data,
    trading_cfg=cfg.trading,
    base_seed=cfg.training.seed,
)
history = ensemble.fit(
    train, val,
    checkpoint_dir=ckpt_dir,
    train_lr=train_lr, val_lr=val_lr,
)
print('\n=== Ensemble training summary ===')
for i, (s, h) in enumerate(zip(history.seeds, history.member_histories)):
    print(f'  member {i:02d} seed={s}: best_val_loss={h.best_val_loss:.5f} '
          f'best_epoch={h.best_epoch}')

## 6. Ensemble inference: mean + epistemic std

In [ ]:
from graduate_work.strategy import (
    AdaptiveConformalPredictor, aci_signals_to_actions,
    ConformalSignalGenerator, SignalGenerator,
    attach_actual_targets, attach_lr_targets,
    build_predictions_frame, calibrate_bayes_threshold,
)

is_cls = cfg.data.mode == 'classification'
test_mean, test_std = ensemble_predict(
    ensemble.members, test['x'],
    batch_size=cfg.training.batch_size, apply_sigmoid=is_cls,
)
val_mean, val_std = ensemble_predict(
    ensemble.members, val['x'],
    batch_size=cfg.training.batch_size, apply_sigmoid=is_cls,
)

print('Per-horizon test prob diagnostics (ensemble):')
for i, h in enumerate(cfg.data.horizons):
    m = test_mean[:, i]
    s = test_std[:, i]
    print(f'  h={h:>2}: mean={m.mean():.4f}  median={np.median(m):.4f}  '
          f'p95={np.quantile(m, 0.95):.4f}  max={m.max():.4f}  '
          f'epistemic_std_mean={s.mean():.4f}')

predictions = build_predictions_frame(
    test['timestamp'], test['ticker'], test_mean, test_std, cfg.data.horizons,
)
val_predictions = build_predictions_frame(
    val['timestamp'], val['ticker'], val_mean, val_std, cfg.data.horizons,
)

## 7. Adaptive Conformal Inference (Gibbs-Candès 2021)

Калибруем на val, потом replay по test'у — α адаптируется онлайн под фактическое покрытие.

In [ ]:
val_targets = attach_actual_targets(val, cfg.data.horizons)
test_targets = attach_actual_targets(test, cfg.data.horizons)

aci = AdaptiveConformalPredictor(
    target_alpha=0.1,    # 90% покрытие positives
    gamma=0.005,         # learning rate из paper'а
    alpha_min=0.001, alpha_max=0.5,
)
aci.calibrate(val_predictions, val_targets)
print('После warm-start:')
print(aci.state_summary.to_string(index=False))

aci_frame = aci.replay(predictions, test_targets)
print('\nПосле online-replay по test:')
print(aci.state_summary.to_string(index=False))
print(f'\nReplay stats: signals_BUY={int((aci_frame["signal"]==1).sum())}, '
      f'miscovered={int(aci_frame["miscovered"].sum())}, '
      f'rows={len(aci_frame)}')

## 8. Бэктест: ACI vs split-conformal vs Bayes

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest

test_prices = test_prices_raw
bars_per_year = cfg.data.bars_per_year
cost_per_trade = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)

# --- 1. ACI signals → backtest ---
aci_signals = aci_signals_to_actions(aci_frame, max_positions=cfg.trading.max_positions)
aci_bt = run_backtest(aci_signals, test_prices, cfg.trading)
aci_m = compute_metrics(aci_bt.equity, aci_bt.trades, periods_per_year=bars_per_year)

# --- 2. Split conformal (директивный) ---
split_gen = ConformalSignalGenerator(cfg.trading, alpha=0.1, directional=True)
split_calib = split_gen.calibrate(val_predictions, val_targets)
split_signals = split_gen.generate(predictions)
split_bt = run_backtest(split_signals, test_prices, cfg.trading)
split_m = compute_metrics(split_bt.equity, split_bt.trades, periods_per_year=bars_per_year)

# --- 3. Bayes-калиброванный порог ---
lr_targets = attach_lr_targets(full_with_targets, val, cfg.data.horizons)
bayes_calib = calibrate_bayes_threshold(
    val_predictions, lr_targets, cost_per_trade=cost_per_trade,
)
tc_bayes = dataclasses.replace(
    cfg.trading,
    probability_threshold=bayes_calib.min_expected_return,
    selection_correction='none',
)
bayes_signals = SignalGenerator(tc_bayes, mode=cfg.data.mode).generate(predictions)
bayes_bt = run_backtest(bayes_signals, test_prices, tc_bayes)
bayes_m = compute_metrics(bayes_bt.equity, bayes_bt.trades, periods_per_year=bars_per_year)

summary = pd.DataFrame([
    {'method': 'ACI',            'threshold': '(adaptive)',
     'n_BUY': int((aci_signals["action"]=="BUY").sum()),
     'sharpe': aci_m['sharpe'],   'total_return_pct': aci_m['total_return']*100,
     'max_dd_pct': aci_m['max_drawdown']*100, 'win_rate': aci_m['win_rate']},
    {'method': 'split-conformal', 'threshold': f'{split_calib.threshold:.4f}',
     'n_BUY': int((split_signals["action"]=="BUY").sum()),
     'sharpe': split_m['sharpe'], 'total_return_pct': split_m['total_return']*100,
     'max_dd_pct': split_m['max_drawdown']*100, 'win_rate': split_m['win_rate']},
    {'method': 'Bayes',           'threshold': f'{bayes_calib.min_expected_return:.4f}',
     'n_BUY': int((bayes_signals["action"]=="BUY").sum()),
     'sharpe': bayes_m['sharpe'], 'total_return_pct': bayes_m['total_return']*100,
     'max_dd_pct': bayes_m['max_drawdown']*100, 'win_rate': bayes_m['win_rate']},
])
print('\n=== iTransformer + Composite + Ensemble + ACI vs alternatives ===')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## 9. Диагностика: эволюция α и threshold по replay

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for h, sub in aci_frame.groupby('horizon'):
    sub = sub.sort_values('timestamp')
    axes[0].plot(sub['timestamp'], sub['alpha'], label=f'h={h}', linewidth=1)
    axes[1].plot(sub['timestamp'], sub['threshold'], label=f'h={h}', linewidth=1)
axes[0].axhline(0.1, ls='--', color='red', alpha=0.5, label='target_alpha=0.1')
axes[0].set_ylabel('alpha (tolerated miscoverage)')
axes[0].set_title('ACI evolution: per-horizon alpha and probability threshold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_ylabel('probability threshold')
axes[1].set_xlabel('timestamp')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
for h, sub in aci_frame.groupby('horizon'):
    sub = sub.sort_values('timestamp')
    rolling_miscov = sub['miscovered'].rolling(window=100, min_periods=20).mean()
    ax.plot(sub['timestamp'], rolling_miscov, label=f'h={h}', linewidth=1)
ax.axhline(0.1, ls='--', color='red', alpha=0.5, label='target=0.1')
ax.set_title('Rolling empirical miscoverage rate (window=100)')
ax.set_ylabel('miscoverage'); ax.set_xlabel('timestamp')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for i, h in enumerate(cfg.data.horizons):
    ax.hist(test_std[:, i], bins=40, alpha=0.5, label=f'h={h}')
ax.set_title('Распределение epistemic std (Deep Ensemble disagreement) по тестовому окну')
ax.set_xlabel('std вероятности по 5 моделям'); ax.set_ylabel('count')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Ансамблевая discrepancy (mean epistemic std) по горизонтам:')
for i, h in enumerate(cfg.data.horizons):
    print(f'  h={h:>2}: epistemic_std={test_std[:, i].mean():.4f}')
print('\nЕсли epistemic_std резко выше на дальних горизонтах — это ожидаемо: '
      'модели расходятся, когда сигнал слабее.')

## 10. Итог

Что мы показали:

1. **iTransformer** обучается на тех же ALGOPACK-данных — variate-attention явно моделирует межканальные зависимости (OFI ↔ spread ↔ imbalance).
2. **Composite loss** (BCE + RankIC + Sharpe + Monotone) оптимизирует именно те метрики, которые важны для торговой стратегии, а не только калибровку.
3. **Deep Ensemble (M=5)** даёт корректную оценку эпистемической неопределённости без MC Dropout-а — каждый член имеет свой локальный минимум.
4. **Adaptive Conformal** обеспечивает формальную гарантию долговременного покрытия и адаптируется к режим-сменам онлайн.

Сравнение с baseline TimeXer'ом (`training_pipeline.ipynb`) и плоским ALGOPACK-baseline'ом (`algopack_baseline.ipynb`) — по тем же метрикам Sharpe / total_return / win_rate / max_dd — даёт прямой ответ на вопрос ВКР: «дают ли SOTA-инструменты прирост качества на премиум-данных MOEX?»